In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
#Data handling Gdelt
DataG1 = pd.read_csv("Datasets/AI/GDELT/deepfake.csv") #deepfake
DataG2 = pd.read_csv("Datasets/AI/GDELT/deepseek.csv") #deepseek
DataG3 = pd.read_csv("Datasets/AI/GDELT/gemini.csv") #gemini
DataG4 = pd.read_csv("Datasets/AI/GDELT/hallucination.csv") #hallucination
DataG5 = pd.read_csv("Datasets/AI/GDELT/llm.csv") #LLm

DataG1 = DataG1.drop("All Articles", axis= 1)
DataG1 = DataG1.rename( columns= {"Article Count" : "deepfake"})

DataG2 = DataG2.drop("All Articles", axis= 1)
DataG2 = DataG2.rename( columns= {"Article Count" : "deepseek"})

DataG3 = DataG3.drop("All Articles", axis= 1)
DataG3 = DataG3.rename( columns= {"Article Count" : "gemini"})

DataG4 = DataG4.drop("All Articles", axis= 1)
DataG4 = DataG4.rename( columns= {"Article Count" : "hallucination"})



In [ ]:
DataG5 = DataG5.pivot(index='Date', columns='Series', values='Value').reset_index()
DataG5 = DataG5.drop("Total Monitored Articles", axis= 1)
DataG5 = DataG5.rename( columns= {"Article Count" : "LLm"})
DataG5 = DataG5.rename( columns= {"Date" : "datetime"})

DataG5['datetime'] = pd.to_datetime(DataG5['datetime'])
DataG5['datetime'] = DataG5['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
DataG5['datetime'] = DataG5['datetime'].str.strip() + '+00:00'
print(DataG5)


In [ ]:
#now all of the words are on a single dataframe like wikipedia
Gdelt = pd.merge(DataG1, DataG2, on = 'datetime')
Gdelt = pd.merge(Gdelt, DataG3, on = 'datetime')
Gdelt = pd.merge(Gdelt, DataG4, on = 'datetime')
Gdelt = pd.merge(Gdelt, DataG5, on = 'datetime')
Gdelt =Gdelt.set_index('datetime')
Gdelt.index = pd.to_datetime(Gdelt.index)
print(Gdelt)
Gdelt.info()

In [ ]:
wiki = pd.read_csv("Datasets/AI/wikiviews/AI_wikiviews.csv")
wiki = wiki.set_index('Date')

wiki['LLM'] = wiki['LLM'].replace('NaN', 0)
wiki['Deepfake'] = wiki['Deepfake'].replace('NaN', 0)
wiki['DeepSeek'] = wiki['DeepSeek'].replace('NaN', 0)
wiki['Gemini'] = wiki['Gemini'].replace('NaN', 0)
wiki['Hallucination (artificial intelligence)'] = wiki['Hallucination (artificial intelligence)'].replace('NaN', 0)
wiki = wiki.rename( columns= {"Hallucination (artificial intelligence)" : "hallucination"})
wiki.index = pd.to_datetime(wiki.index)
wiki.columns = wiki.columns.str.lower()
print(wiki)

In [ ]:
Gdelt = Gdelt[Gdelt.index <= '2026-02-19']
wiki = wiki[wiki.index >= '2017-01-01']
wiki = wiki[wiki.index <= '2026-02-19']

In [ ]:
#Offerta is supply
Gdelt['Offerta'] = Gdelt.sum(axis=1)
Gdelt = Gdelt.drop(columns="deepfake")
Gdelt = Gdelt.drop(columns="deepseek")
Gdelt = Gdelt.drop(columns="gemini")
Gdelt = Gdelt.drop(columns="hallucination")
Gdelt = Gdelt.drop(columns="LLm")
#substitute the keyword with your own

print(Gdelt)

In [ ]:
Gdelt = Gdelt/Gdelt.mean()
wiki = wiki / wiki.mean()
AI_dataset = wiki.add(Gdelt, fill_value=0)

In [ ]:
AI_dataset['deltalog'] = np.log((AI_dataset['Offerta']+1)/(AI_dataset['Domanda'] +1) )
AI_dataset['deltalog'].plot(ylabel='delta log', xlabel= 'date')
print(AI_dataset)

In [ ]:
AI_dataset.to_csv("AI_dataset.csv", index= True)